In [24]:
from openai import OpenAI 
import json
import networkx as nx
import ipycytoscape
import pandas as pd
import os
import math
import re
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore", category= DeprecationWarning)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)


In [25]:
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
llm_model_name = "deepseek-ai/DeepSeek-V3"

In [26]:
load_dotenv()

LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")


client = OpenAI(
    # Sets the base URL for the API endpoint to the Nebius service.
    base_url=LITELLM_API_BASE,
    # Sets the API key for authentication. Replace with your actual key, preferably loaded from a secure source.
    api_key=NEBIUS_API_KEY
)

In [27]:
llm_temperature = 0.0
llm_max_tokens = 4096

In [36]:
unstructured_text = """
For the history of the British Isles before the formation of the United Kingdom, see History of the British Isles.
"UK History" redirects here. For the UKTV channel formerly known as UK History, see U&Yesterday.

A published version of the Treaty of Union agreement, which led to the creation of the Kingdom of Great Britain in 1707
This article is part of a series on the
History of the
United Kingdom

Timeline
Topics
Places
OutlineList of yearsHistoriography Category Portal
vte
The history of the United Kingdom begins in 1707 with the Treaty of Union and Acts of Union. The core of the United Kingdom as a unified state came into being with the political union of the kingdoms of England and Scotland,[1] into a new unitary state called Great Britain.[a]

The first decades were marked by Jacobite risings which ended with defeat for the Stuart cause at the Battle of Culloden in 1746. In 1763, victory in the Seven Years' War led to the growth of the First British Empire. With defeat by the US, France and Spain in the War of American Independence, Great Britain lost its 13 American colonies and rebuilt a Second British Empire based in Asia and Africa. Politically the central event was the French Revolution and its Napoleonic aftermath from 1793 to 1815, which British elites saw as a profound threat, and worked energetically to form multiple coalitions that finally defeated Napoleon in 1815. The Acts of Union 1800 added the Kingdom of Ireland to create the United Kingdom of Great Britain and Ireland.

The Tories, who came to power in 1783, remained in power until 1830. Forces of reform opened decades of political reform that broadened the ballot, and opened the economy to free trade. The outstanding political leaders of the 19th century included Palmerston, Disraeli, Gladstone, and Salisbury. Culturally, the Victorian era was a time of prosperity and dominant middle-class virtues when Britain dominated the world economy and maintained a generally peaceful century from 1815 to 1914. The First World War, with Britain in alliance with France, Russia and the US, was a furious but ultimately successful total war with Germany. The resulting League of Nations was a favourite project in Interwar Britain. In 1922, 26 counties of Ireland seceded to become the Irish Free State; a day later, Northern Ireland seceded from the Free State and returned to the United Kingdom. In 1927, the United Kingdom changed its formal title to the United Kingdom of Great Britain and Northern Ireland,[2] usually shortened to Britain, United Kingdom or UK. While the Empire remained strong, as did the London financial markets, the British industrial base began to slip behind Germany and the US. Sentiments for peace were so strong that the nation supported appeasement of Hitler's Germany in the 1930s, until the Nazi invasion of Poland in 1939 started the Second World War. In the Second World War, the Soviet Union and the US joined the UK as the main Allied powers.

After the war, Britain was no longer a military or economic superpower, as seen in the Suez Crisis of 1956. Britain granted independence to almost all its possessions. The new states typically joined the Commonwealth of Nations. The postwar years saw great hardships, alleviated somewhat by large-scale financial aid from the US. Prosperity returned in the 1950s. Meanwhile, from 1945 to 1950, the Labour Party built a welfare state, nationalised many industries, and created the National Health Service. The UK took a strong stand against Communist expansion after 1945, playing a major role in the Cold War and the formation of NATO as an anti-Soviet military alliance with West Germany, France, the US, Italy, Canada and smaller countries. The UK has been a leading member of the United Nations since its founding, as well as other international organisations. In the 1990s, neoliberalism led to the privatisation of nationalised industries and significant deregulation of business affairs. London's status as a world financial hub grew. Since the 1990s, large-scale devolution movements in Northern Ireland, Scotland and Wales have decentralised political decision-making. Britain has moved back and forth on its economic relationships with Western Europe. It joined the European Economic Community in 1973, thereby weakening economic ties with its Commonwealth. However, the Brexit referendum in 2016 committed the UK to leave the European Union, which it did in 2020.
"""

In [30]:
char_count = len(unstructured_text)
word_count = len(unstructured_text.split())

print("Word count", word_count)
print("Char count", char_count)

print("Unstructured Text \n", unstructured_text)

Word count 934
Char count 5429
Unstructured Text 
 
THe beginning of Nations, those excepted of whom sacred Books have spok'n, is to this day unknown. Nor only the beginning, but the deeds also of many succeeding Ages, yea periods of Ages, either wholly unknown, or obscur'd and blemisht with Fables. Whether it were that the use of Letters came in long after, or were it the violence of barbarous inundations, or they themselves at certain revolutions of time, fatally decaying, and degenerating into Sloth and Ignorance; wherby the monuments of more ancient civility have bin som destroy'd, som lost. Perhaps dis-esteem and contempt of the public affairs then present, as not worth recording, might partly be in cause. Certainly oft-times we see that wise men, and of best abilitie, have forborn to write the Acts of thir own daies, while they beheld with a just loathing and disdain, not only how unworthy, how pervers, how corrupt, but often how ignoble, how petty, how below all History the pers

In [31]:
#Chunking Configuration

chunk_size = 150
chunk_overlap = 30

if chunk_overlap > chunk_size and chunk_size > 0:
    raise SystemExit(f"Erorr: Overlap {chunk_overlap} must be smaller than chunk size {chunk_size}")
else:
    print("Chunnking condig is valid")

Chunnking condig is valid


In [33]:
words = unstructured_text.split()
total_words = len(words)

In [34]:
chunks = []
start_index = 0
chunk_number = 1

while start_index < total_words:
    end_index = min(start_index + chunk_size, total_words)
    chunk_text = " ".join(words[start_index: end_index])
    chunks.append({"text":chunk_text, "chunk_number": chunk_number})

    next_start_index = start_index + chunk_size - chunk_overlap

    if next_start_index <= start_index:
        if end_index == total_words:
            break
        next_start_index = start_index + 1

    start_index = next_start_index
    chunk_number += 1

    if chunk_number > total_words:
        break
    


In [35]:
if chunks:
    chunks_df = pd.DataFrame(chunks)
    chunks_df["word_count"] = chunks_df["text"].apply(lambda x: len(x.split()))
    display(chunks_df[["chunk_number", "word_count", "text"]])

else:
    print("No chunks were created")
    

,chunk_number,word_count,text
0,1,150,"THe beginning of Nations, those excepted of whom sacred Books have spok'n, is to this day unknown. Nor only the beginning, but the deeds also of m..."
1,2,150,"abilitie, have forborn to write the Acts of thir own daies, while they beheld with a just loathing and disdain, not only how unworthy, how pervers..."
2,3,150,"both privat, and public, among which well may History be reck'nd, they us'd the Greek Tongue: and that the British Druids who taught those in Gaul..."
3,4,150,"judicious Antiquaries bin long rejected for a modern Fable. Nevertheless there being others, besides the first suppos'd Author, men not unread, no..."
4,5,150,"them judiciously. I might also produce example, as Diodorus among the Greeks, Livie and others of the Latines, Polydore and Virunnius accounted am..."
5,6,150,"That the whole Earth was inhabited before the Flood, and to the utmost point of habitable ground, from those effectual words of God in the Creatio..."
6,7,150,"with all circumstance they tell us when, and who first set foot upon this Iland, presume to name out of fabulous and counterfet Authors a certain ..."
7,8,94,"may easily excuse our not allowing it the room heer so much as of a British Fable. That which follows, perhaps as wide from truth, though seeming ..."


## SYSTEM AND USER PROMPTS 

In [37]:
extraction_system_prompt = """
You are an AI expert specialized in knowledge graph extraction. Your task is to identify and extract factual Subject-Predicate-Object (SPO) triples from the given text. 
Focus on accuracy and adhere strictly to JSON output format request in the user prompt. Extract core entities and the most direct relationship
"""

extraction_user_prompt_template = """

Please extract Subject-Predicate-Object (SPO) triples from the text below.

**VERY IMPORTANT RULES:**
1.  **Output Format:** Respond ONLY with a single, valid JSON array. Each element MUST be an object with keys "subject", "predicate", "object".
2.  **JSON Only:** Do NOT include any text before or after the JSON array (e.g., no 'Here is the JSON:' or explanations). Do NOT use markdown ```json ... ``` tags.
3.  **Concise Predicates:** Keep the 'predicate' value concise (1-3 words, ideally 1-2). Use verbs or short verb phrases (e.g., 'discovered', 'was born in', 'won').
4.  **Lowercase:** ALL values for 'subject', 'predicate', and 'object' MUST be lowercase.
5.  **Pronoun Resolution:** Replace pronouns (she, he, it, her, etc.) with the specific lowercase entity name they refer to based on the text context (e.g., 'United Kingdom').
6.  **Specificity:** Capture specific details (e.g., 'Island in the Europe' instead of just 'Island' if specified).
7.  **Completeness:** Extract all distinct factual relationships mentioned.

**Text to Process:**
```text
{text_chunk}
```

**Required JSON Output Format Example:**
[
  {{ "subject": "Tories", "predicate": "rules", "object": "United Kingdom" }},
  {{ "subject": "Labour Party", "predicate": "built", "object": "welfare state" }}
]

**Your JSON Output (MUST start with '[' and end with ']'):**
"""

In [38]:
all_extracted_triples = []
failed_chunks = []

In [39]:
chunk_index = 0

if chunk_index < len(chunks):
    chunk_info = chunks[chunk_index]
    chunk_text = chunk_info["text"]
    chunk_num = chunk_info["chunk_number"]

    user_prompt = extraction_user_prompt_template.format(text_chunk = chunk_text)
    llm_output = None
    error_message = None

    try:
        response = client.chat.completions.create(
            model = llm_model_name,
            messages = [
                {"role": "system", "content": extraction_system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        llm_output = response.choices[0].message.content.strip()
        print(llm_output)

    except Exception as e:
        error_message = str(e)
        failed_chunks.append({"chunk_number": chunk_num, "error": f"API/Processing error: {error_message}", "response": " "})

    parsed_json = None
    parsing_error = None
    if llm_output is not None:
        try:
            parsed_data = json.loads(llm_output)

            if isinstance(parsed_data, dict):
                list_values = [v for v in parsed_data.values() if isinstance(v,list)]
                if len(list_values) == 1:
                    parsed_json = list_values[0]
                else:
                    raise ValueError("JSON object received, but does not contain single list of triples") 
            elif isinstance(parsed_data, list):
                parsed_json = parsed_data
            else:
                raise ValueError("Parsed JSON is not a list or expected dictionary wrapper")

        except json.JSONDecodeError as json_error:
            parsing_error = f"JSONDecodeError: {json_error}. Trying regex fallback..."
            match = re.search(r'^\s*(\[.*?\])\s*$', llm_output, re.DOTALL)
            if match:
                json_string_extracted = match.group(1)

                try:
                    parsed_json = json.loads(json_string_extracted)
                    parsing_error = None
                except json.JSONDecodeError as nested_err:
                    print("regex content is not valid error")
                    print(f"      ERROR: Regex content is not valid JSON: {nested_err}")

            else:
                parsing_error = "JSONDecodeError and Regex fallback failed."
        
        except ValueError as val_err:
            parsing_error = f"Value Error: {val_err}"


    if parsed_json is not None:
        valid_triples_in_chunk = []
        invalid_entries = []
        if isinstance(parsed_json, list):
            for item in parsed_json:
                if isinstance(item, dict) and all(k in item for k in ["subject", "predicate", "object"]):
                    if all(isinstance(item[k], str) for k in ['subject', 'predicate', 'object']):
                        item['chunk'] = chunk_num
                        valid_triples_in_chunk.append(item)
                    else:
                        invalid_entries.append({"item": item, "reason": "Non-string value"})
                else:
                    invalid_entries.append({"item": item, "reason": "Incorrect structure/ keys"})
        
        else:
            invalid_entries({"item" : parsed_json, "reason": "Not a list"})

            if not any(fc["chunk_number"] == chunk_num for fc in failed_chunks):
                failed_chunks.append({"chunk_number": chunk_num, "error": "Parsed data not a list", "response": llm_output})

        if invalid_entries:
            print(f"Skipped {len(invalid_entries)} invalid entries")

        if valid_triples_in_chunk:
            display(pd.DataFrame(valid_triples_in_chunk))
            all_extracted_triples.extend(valid_triples_in_chunk)
        
        else:
            print("sdfsd")

else:
    print(f"chunk index {chunk_index} is out of bounds: (Total CHUkns): {len(chunks)}")


## Normalize and De-deplicate Triples

In [40]:
normalized_triples = []
seen_triples = set()
original_count = len(all_extracted_triples)
empty_removed_count = 0
duplicates_removed_count = 0

In [41]:

example_limit = 5
processed_count = 0

for i,triple in enumerate(all_extracted_triples):
    show_example = (i<example_limit)
    if show_example:
        print(f" Example {i+1}")

    subject_raw = triple.get("subject")
    predicate_raw = triple.get("predicate")
    object_raw = triple.get("object")
    chunk_num = triple.ge("chunk", "unknown")

    triple_valid = False

    normalized_sub, normalized_pred, normalized_obj = None, None, None

    if isinstance(subject_raw, str) and isinstance(predicate_raw, str) and isinstance(object_raw, str):
        normalized_sub = subject_raw.strip().lower()
        normalized_pred = re.sub(f'\s+', '', predicate_raw.strip().lower).strip()
        normalized_obj = object_raw.strip().lower()

        if normalized_sub and normalized_obj and normalized_pred:
            triple_identifier = (normalized_sub, normalized_obj, normalized_pred)

            if triple_identifier not in seen_triples:
                normalized_triples.append({
                    'subject': normalized_sub,
                    'predicate': normalized_pred,
                    'object': normalized_obj,
                    'source_chunk': chunk_num

                })
                seen_triples.add(triple_identifier)
                triple_valid = True
            
            else:
                duplicates_removed_count += 1
    
    else:
        empty_removed_count += 1
    
    processed_count += 1
        

<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
/var/folders/n4/q09qt1ds0wdf7rdtzg2f5xm40000gn/T/ipykernel_88769/1025485936.py:20: SyntaxWarning: invalid escape sequence '\s'
  normalized_pred = re.sub(f'\s+', '', predicate_raw.strip().lower).strip()
